In [ ]:
pip show torch


In [ ]:
pip install --upgrade torch


In [ ]:
pip install --upgrade torchvision


In [ ]:
pip install torch==1.12.0 torchvision==0.13.0


In [ ]:
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)


In [ ]:
import os
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score
import cv2
import numpy as np

# ✅ Ensure the correct image directory
img_dir = "/bin/textures"

if not os.path.exists(img_dir):
    raise FileNotFoundError(f"❌ Image directory '{img_dir}' does not exist!")

# ✅ Define the expected filenames
expected_filenames = [
    "data01.jpeg", "data02.jpg", "data03.jpg", "data04.jpg", "data05.jpg", "data06.jpg",
    "data_07.jpg", "data_08.jpg", "data_09.jpg", "data_10.jpg", "data_11.jpg", "data_12.jpg",
    "data_13.jpg", "data_14.jpg", "data_15.jpg", "data_16.jpg", "data_17.jpg", "data_18.jpg",
    "data_19.jpg", "data_20.jpg"
]

# ✅ Check if all expected files exist
missing_files = [f for f in expected_filenames if not os.path.exists(os.path.join(img_dir, f))]

if missing_files:
    raise FileNotFoundError(f"❌ Missing Images: {missing_files}")
else:
    print(f"✅ All {len(expected_filenames)} images are found in '{img_dir}'!")


# Custom Dataset Class
class ArtisticDataset(Dataset):
    def __init__(self, img_dir, img_list, transform=None):
        self.img_dir = img_dir
        self.img_names = img_list
        self.transform = transform

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_name = self.img_names[idx]
        img_path = os.path.join(self.img_dir, img_name)

        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)

        label = idx % 5  # Assign labels (0-4)
        return img, label


# Define Transformations
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
dataset = ArtisticDataset(img_dir=img_dir, img_list=expected_filenames, transform=transform)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# Check dataset loading
for images, labels in dataloader:
    print(f"Images shape: {images.shape}")
    print(f"Labels: {labels}")
    break  # Check only the first batch


# Load a pre-trained ResNet-18 model
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Modify the final fully connected layer
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 5)  # Assuming 5 classes (0-4)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Set loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10
train_losses = []
train_accuracies = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_predictions = 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct_predictions += (predicted == labels).sum().item()
        total_predictions += labels.size(0)

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = correct_predictions / total_predictions
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")

    # Save model checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        torch.save(model.state_dict(), f"model_checkpoint_epoch_{epoch+1}.pth")


# Evaluate Model
model.eval()
true_labels = []
pred_labels = []

with torch.no_grad():
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        true_labels.extend(labels.cpu().numpy())
        pred_labels.extend(predicted.cpu().numpy())

# Compute accuracy and F1 score
accuracy = accuracy_score(true_labels, pred_labels)
f1 = f1_score(true_labels, pred_labels, average='weighted')

print(f"Validation Accuracy: {accuracy:.4f}")
print(f"Validation F1 Score: {f1:.4f}")


# Plot Training Loss and Accuracy
plt.figure(figsize=(12, 6))

# Loss Curve
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training Loss')

# Accuracy Curve
plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label='Training Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training Accuracy')

plt.show()


# Grad-CAM Visualization
def generate_gradcam(model, image):
    """Generate Grad-CAM heatmap for a given image"""
    model.eval()

    def save_gradient(grad):
        nonlocal gradient
        gradient = grad

    target_layer = model.layer4[1].conv2
    target_layer.register_full_backward_hook(save_gradient)

    image = image.unsqueeze(0).to(device)
    output = model(image)
    class_idx = torch.argmax(output, dim=1).item()

    model.zero_grad()
    output[0, class_idx].backward()

    activations = model.layer4[1].conv2.weight
    gradients = gradient

    pooled_gradients = torch.mean(gradients, dim=[0, 2, 3])
    for i in range(activations.size(1)):
        activations[0, i, :, :] *= pooled_gradients[i]

    heatmap = torch.mean(activations, dim=1).squeeze().cpu().detach().numpy()
    heatmap = np.maximum(heatmap, 0)
    heatmap /= np.max(heatmap)

    heatmap = cv2.resize(heatmap, (224, 224))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    img = image.squeeze(0).permute(1, 2, 0).cpu().numpy()
    img = (img - img.min()) / (img.max() - img.min())  # Normalize
    img = np.uint8(255 * img)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

    overlay = cv2.addWeighted(img, 0.7, heatmap, 0.3, 0)
    return overlay


# Visualize Grad-CAM
sample_image, _ = dataset[0]
overlay = generate_gradcam(model, sample_image)

plt.imshow(overlay)
plt.title("Grad-CAM Overlay")
plt.show()
